<a href="https://colab.research.google.com/github/dystaSatria/Deep-Learning/blob/main/Internship%20Projects/Istatisik_DenseNet/istatistik.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix

# Veri yükleme ve normalizasyon (istatistiksel ön işleme)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # Ortalama ve standart sapma ile standardizasyon
])

train_data = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)

# Basit bir DenseNet Bloğu
class DenseLayer(nn.Module):
    def __init__(self, in_channels, growth_rate):
        super().__init__()
        self.conv = nn.Sequential(
            nn.BatchNorm2d(in_channels),
            nn.ReLU(),
            nn.Conv2d(in_channels, growth_rate, kernel_size=3, padding=1)
        )

    def forward(self, x):
        return torch.cat([x, self.conv(x)], dim=1)

# Model tanımı
class DenseNet(nn.Module):
    def __init__(self, in_channels=1, growth_rate=32):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3),
            nn.MaxPool2d(kernel_size=3, stride=2),
            DenseLayer(64, growth_rate),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(64 + growth_rate, 10)  # MNIST için 10 sınıf
        )

    def forward(self, x):
        return self.features(x)

# Eğitim döngüsü ve istatistiksel metrikler
model = DenseNet()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(5):
    for images, labels in train_loader:
        outputs = model(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # Test verisi üzerinde istatistiksel doğruluk
    with torch.no_grad():
        predictions = model(images).argmax(dim=1)
        accuracy = accuracy_score(labels.numpy(), predictions.numpy())
        conf_matrix = confusion_matrix(labels.numpy(), predictions.numpy())
        print(f"Epoch {epoch+1}, Doğruluk: {accuracy:.4f}, Kayıp: {loss.item():.4f}")
        print("Karışıklık Matrisi:\n", conf_matrix)

100%|██████████| 9.91M/9.91M [00:00<00:00, 11.3MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 342kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.10MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.58MB/s]


Epoch 1, Doğruluk: 0.9688, Kayıp: 0.1979
Karışıklık Matrisi:
 [[3 0 0 0 0 0 0 0 0 0]
 [0 6 1 0 0 0 0 0 0 0]
 [0 0 2 0 0 0 0 0 0 0]
 [0 0 0 1 0 0 0 0 0 0]
 [0 0 0 0 2 0 0 0 0 0]
 [0 0 0 0 0 3 0 0 0 0]
 [0 0 0 0 0 0 3 0 0 0]
 [0 0 0 0 0 0 0 5 0 0]
 [0 0 0 0 0 0 0 0 2 0]
 [0 0 0 0 0 0 0 0 0 4]]
Epoch 2, Doğruluk: 1.0000, Kayıp: 0.0351
Karışıklık Matrisi:
 [[5 0 0 0 0 0 0 0 0 0]
 [0 2 0 0 0 0 0 0 0 0]
 [0 0 2 0 0 0 0 0 0 0]
 [0 0 0 5 0 0 0 0 0 0]
 [0 0 0 0 2 0 0 0 0 0]
 [0 0 0 0 0 1 0 0 0 0]
 [0 0 0 0 0 0 2 0 0 0]
 [0 0 0 0 0 0 0 6 0 0]
 [0 0 0 0 0 0 0 0 5 0]
 [0 0 0 0 0 0 0 0 0 2]]
Epoch 3, Doğruluk: 0.8750, Kayıp: 0.3915
Karışıklık Matrisi:
 [[2 0 0 0 0 0 0 0 0 0]
 [0 1 0 0 0 0 0 0 0 0]
 [0 0 2 0 0 0 0 0 0 0]
 [0 0 0 2 0 0 0 0 0 0]
 [0 0 0 0 4 0 0 0 0 0]
 [0 1 0 0 0 6 0 1 0 0]
 [0 0 1 0 0 0 3 0 0 0]
 [0 0 0 0 0 0 0 3 0 0]
 [1 0 0 0 0 0 0 0 2 0]
 [0 0 0 0 0 0 0 0 0 3]]
Epoch 4, Doğruluk: 1.0000, Kayıp: 0.0721
Karışıklık Matrisi:
 [[5 0 0 0 0 0 0 0 0 0]
 [0 7 0 0 0 0 0 0 0 0]
 [0 0 3 0 0 0